# Hypothesis 3 — Colab / 实体服务器 Jupyter
流程：**配置 → 跑题 → 评价 → 下载/打包**。

可在已有 GPU 服务器环境运行，不强制依赖 /content 或 google.colab。
依赖先检测；满足版本范围的直接复用，缺少或不兼容时才调用一次 pip。
PyTorch 由运行环境提供并锁定版本，不自动安装/升级 PyTorch 或 CUDA。

模型先检查指定本地目录与 HF 缓存，完整缓存命中不联网、不重复下载权重。
只有缺失/不完整的模型文件才交给 HF 增量下载；可设 ALLOW_MODEL_DOWNLOADS=False 严格离线。
服务器可设置 HF_CACHE_DIR 为持久磁盘目录；None 会复用 HF_HUB_CACHE/HF_HOME/默认缓存。
Colab shutdown 后临时盘缓存可能丢失；本 notebook 不访问 Google Drive。

默认排队 22 个本地模型，Jev API 跳过；各模型单独进程运行，失败继续。
Laya 每 0.25 秒监听 RAM/GPU 显存，连续 64 次低于 70% 才扩批。
其他原生模型/语言模型基线使用独立适配器，尚待相应服务器实测。
NanoJev 根据当前 Torch 接口决定是否调用 native Triton 注销函数，不要求重装 Torch。

单选主报去随机基线分和区间；多选另报集合指标/基线，保留失败和模型覆盖表。
服务器端生成 ZIP 并显示下载链接；Colab 端调用浏览器下载。


In [ ]:
# 1. 配置：复用环境、模型缓存和已生成题库
import os, sys, json, subprocess, hashlib, importlib.util
from pathlib import Path
from importlib.metadata import version
from datetime import datetime, timezone
from uuid import uuid4

# 服务器从仓库目录或 notebooks/ 打开时自动定位源码；也可手动设置 REPO_PATH。
REPO_PATH = None
WORK_ROOT_PATH = None  # None: Colab 用 /content；服务器用仓库 data/hypothesis3_v1/runtime
HF_CACHE_DIR = None  # 例如 "/srv/cache/huggingface/hub"；None: 保留服务器原有 HF 缓存设置
AUTO_INSTALL_MISSING = True
ALLOW_MODEL_DOWNLOADS = True  # False: 只读本地目录/完整缓存，缺少文件标记 BLOCKED_CACHE
ALLOW_DATA_DOWNLOADS = True   # False: 题库未生成时只用已缓存的原始数据
UPDATE_REPO = False           # 已有源码默认不自动 git pull，保留服务器本地修改
MODELS = "all"                # 或 ["nanojev", "kev_08b", ...]
MODEL_OVERRIDES = {}          # 支持 path/base_path/revision/base_revision/precision
PROFILE = "full"
SCHEDULER = "adaptive"
ADAPTIVE = {
    "initial_batch_rows": 8, "max_batch_rows": 128, "max_padded_tokens": 32768,
    "producer_workers": 2, "prefetch_cases": 64, "lookahead_cases": 64,
    "sample_interval": 0.25, "low_samples": 64, "grow_below": 0.70,
    "shrink_above": 0.85, "reserve_gib": 2.0,
}
BOOTSTRAP_REPS = 2000
EVALUATION_SEED = 20261026
EXCLUSION_FILES = []

try:
    IN_COLAB = Path("/content").is_dir() and importlib.util.find_spec("google.colab") is not None
except (ImportError, ModuleNotFoundError):
    IN_COLAB = False
assert sys.version_info >= (3, 12), "需要 Python 3.12+；请在服务器选择相应 kernel"
assert PROFILE in ("full", "pilot")
if REPO_PATH is not None:
    REPO = Path(REPO_PATH).expanduser().resolve()
else:
    candidates = [Path.cwd(), *Path.cwd().parents]
    REPO = next((p for p in candidates if (p / "research/hypothesis3_v1/run.py").is_file()), None)
    if REPO is None:
        REPO = (Path("/content") if IN_COLAB else Path.cwd()) / "system1bench"
WORK_ROOT = (Path(WORK_ROOT_PATH).expanduser().resolve() if WORK_ROOT_PATH is not None
             else Path("/content") if IN_COLAB else REPO.parent / "hypothesis3-runtime")
WORK_ROOT.mkdir(parents=True, exist_ok=True)
SESSION = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S") + "-" + uuid4().hex[:6]
LOG_DIR = WORK_ROOT / "hypothesis3-logs" / SESSION
LOG_DIR.mkdir(parents=True, exist_ok=True)
COMMAND_NUMBER = 0

def run(args, cwd=None):
    global COMMAND_NUMBER
    args = list(map(str, args))
    COMMAND_NUMBER += 1
    log = LOG_DIR / f"command-{COMMAND_NUMBER:03d}.log"
    print("+", " ".join(args), "\n日志:", log, flush=True)
    tail = []
    with log.open("w", encoding="utf-8") as stream:
        process = subprocess.Popen(args, cwd=cwd, stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace", bufsize=1)
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
                stream.write(line)
                stream.flush()
                tail.append(line)
                tail = tail[-40:]
            code = process.wait()
        except BaseException:
            process.terminate()
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            raise
    if code:
        raise RuntimeError(f"退出码 {code}；日志 {log}\n{''.join(tail)}")

if not REPO.exists():
    run(["git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
         "https://github.com/Jaffe2718/system1bench.git", REPO])
    run(["git", "sparse-checkout", "set", "system1bench", "research/hypothesis3_v1",
         "research/model_expansion_v2", "research/model_expansion_v3",
         "research/startlux_transfer_v1"], cwd=REPO)
elif UPDATE_REPO:
    run(["git", "pull", "--ff-only", "origin", "main"], cwd=REPO)
EXPERIMENT = REPO / "research/hypothesis3_v1"
SUPPORT_PATHS = ["research/model_expansion_v2", "research/model_expansion_v3",
                 "research/startlux_transfer_v1"]
if (REPO / ".git/info/sparse-checkout").exists() and any(not (REPO / p).is_dir() for p in SUPPORT_PATHS):
    run(["git", "sparse-checkout", "add", "research/hypothesis3_v1", *SUPPORT_PATHS], cwd=REPO)
for required in ("run.py", "models.py", "environment.py", "evaluate.py", "requirements-notebook.txt"):
    assert (EXPERIMENT / required).is_file(), f"源码缺少 {required}，请先同步本地仓库"
PY = Path(sys.executable)

# 主机已有的 PyTorch 必须可用；不由 notebook 下载或替换它。
import torch
assert torch.cuda.is_available(), "需要已有 CUDA PyTorch/GPU 环境；请让服务器管理员准备，不自动重装"
TORCH_VERSION = version("torch")
print("保留 PyTorch:", TORCH_VERSION, "GPU:", torch.cuda.get_device_name(0))
if HF_CACHE_DIR is not None:
    HF_CACHE_DIR = str(Path(HF_CACHE_DIR).expanduser().resolve())
    os.environ["HF_HUB_CACHE"] = HF_CACHE_DIR
CONSTRAINT = WORK_ROOT / "hypothesis3-torch-constraint.txt"
CONSTRAINT.write_text("torch==" + TORCH_VERSION, encoding="utf-8")

env_spec = importlib.util.spec_from_file_location("h3_environment", EXPERIMENT / "environment.py")
env_check = importlib.util.module_from_spec(env_spec)
env_spec.loader.exec_module(env_check)
missing, reused = env_check.missing_requirements(EXPERIMENT / "requirements-notebook.txt")
print("复用已有依赖:", ", ".join(f"{p['name']}=={p['version']}" for p in reused))
if missing:
    print("需补齐/调整:", missing)
    if not AUTO_INSTALL_MISSING:
        raise RuntimeError("AUTO_INSTALL_MISSING=False；请先在服务器准备以上依赖")
    run([PY, "-m", "pip", "install", "--disable-pip-version-check",
         "--constraint", CONSTRAINT, *missing], cwd=REPO)
else:
    print("依赖均满足要求，跳过 pip install。")
run([PY, "-c",
     "import sys, torch, numpy, pyarrow.parquet, laya, psutil, peft, pydantic, accelerate, bitsandbytes; "
     "from importlib.metadata import version; "
     "assert version('torch') == sys.argv[1]; print('所需模块导入成功，PyTorch 未变')",
     TORCH_VERSION], cwd=REPO)

registry_spec = importlib.util.spec_from_file_location("h3_registry", EXPERIMENT / "models.py")
registry = importlib.util.module_from_spec(registry_spec)
registry_spec.loader.exec_module(registry)
registry.configure_storage(HF_CACHE_DIR, not ALLOW_MODEL_DOWNLOADS)
if MODELS == "all":
    MODELS = registry.model_names()
else:
    MODELS = registry.select_models(MODELS)
MODEL_CONFIG = WORK_ROOT / f"hypothesis3-model-overrides-{SESSION}.json"
MODEL_CONFIG.write_text(json.dumps(MODEL_OVERRIDES, ensure_ascii=False, indent=2), encoding="utf-8")
from huggingface_hub.constants import HF_HUB_CACHE
print("HF 缓存:", HF_CACHE_DIR or HF_HUB_CACHE, "允许下载:", ALLOW_MODEL_DOWNLOADS)
print("本次模型:", MODELS)

BANK_REVISION = hashlib.sha256(
    (EXPERIMENT / "build.py").read_bytes() + (EXPERIMENT / "sources.json").read_bytes()).hexdigest()[:12]
BANK = WORK_ROOT / "hypothesis3-banks" / BANK_REVISION
CACHE = WORK_ROOT / "hypothesis3-cache/raw"
if EXCLUSION_FILES:
    extra = hashlib.sha256(b"".join(Path(p).read_bytes() for p in EXCLUSION_FILES)).hexdigest()[:12]
    BANK = BANK.with_name(BANK.name + "-" + extra)
if BANK.exists():
    run([PY, EXPERIMENT / "build.py", "--output", BANK, "--verify"], cwd=REPO)
else:
    args = [PY, "-u", EXPERIMENT / "build.py", "--cache", CACHE, "--output", BANK]
    if not ALLOW_DATA_DOWNLOADS:
        args += ["--offline"]
    for filename in EXCLUSION_FILES:
        args += ["--exclude", filename]
    run(args, cwd=REPO)
if not EXCLUSION_FILES:
    receipt = json.loads((EXPERIMENT / "bank_receipt.json").read_text(encoding="utf-8"))
    assert hashlib.sha256((BANK / "manifest.json").read_bytes()).hexdigest() == receipt["manifest_sha256"]
    for filename, expected in receipt["files"].items():
        assert hashlib.sha256((BANK / filename).read_bytes()).hexdigest() == expected["sha256"], filename
    print("题库与冻结回执一致。")

bank_manifest = json.loads((BANK / "manifest.json").read_text(encoding="utf-8"))
panel_budget = {panel: bank_manifest["files"][panel + ".jsonl"] for panel in ("native", "paired", "multi_select")}
if PROFILE == "full":
    per_model_requests = sum(p["requests"] for p in panel_budget.values())
    per_model_decisions = sum(p["decisions"] for p in panel_budget.values())
    mother_questions = sum(p["bases"] for p in panel_budget.values())
else:
    pilot_bases = set(json.loads((BANK / "pilot_base_ids.json").read_text(encoding="utf-8")))
    pilot_rows = [json.loads(line) for line in (BANK / "paired.jsonl").read_text(encoding="utf-8").splitlines()]
    pilot_rows = [r for r in pilot_rows if r["base_id"] in pilot_bases]
    mother_questions, per_model_requests = len(pilot_bases), len(pilot_rows)
    per_model_decisions = sum(len(r["questions"]) for r in pilot_rows)
REQUEST_BUDGET = {"mother_questions": mother_questions, "requests_per_model": per_model_requests,
                  "decisions_per_model": per_model_decisions,
                  "total_requests_if_all_models_complete": per_model_requests * len(MODELS)}
print("独立母题:", mother_questions, "每模型请求:", per_model_requests,
      "每模型决策:", per_model_decisions, "全模型请求预算:", REQUEST_BUDGET["total_requests_if_all_models_complete"])

OUTPUT = WORK_ROOT / "hypothesis3-results" / SESSION
OUTPUT.mkdir(parents=True, exist_ok=True)
(OUTPUT / "configuration.json").write_text(json.dumps({
    "models": MODELS, "model_overrides": MODEL_OVERRIDES, "profile": PROFILE, "request_budget": REQUEST_BUDGET,
    "bank": str(BANK), "scheduler": SCHEDULER, "adaptive": ADAPTIVE,
    "evaluation_planned": True, "bootstrap_repetitions": BOOTSTRAP_REPS,
    "evaluation_seed": EVALUATION_SEED, "hf_cache_dir": HF_CACHE_DIR or HF_HUB_CACHE,
    "model_downloads_allowed": ALLOW_MODEL_DOWNLOADS, "platform_colab": IN_COLAB,
    "reused_dependencies": reused, "installed_requirements": missing,
}, ensure_ascii=False, indent=2), encoding="utf-8")
print("配置完成。输出:", OUTPUT)


In [ ]:
# 2. 跑题：本地完整缓存优先，缺失才下载；单模型失败后继续
scheduler_args = ["--scheduler", SCHEDULER]
for key, value in ADAPTIVE.items():
    scheduler_args += ["--" + key.replace("_", "-"), str(value)]
storage_args = ["--cache-dir", HF_CACHE_DIR] if HF_CACHE_DIR else []
if not ALLOW_MODEL_DOWNLOADS:
    storage_args += ["--offline"]
run([PY, "-u", EXPERIMENT / "run.py", "--bank", BANK, "--output", OUTPUT,
     "--models", *MODELS, "--profile", PROFILE, "--model-config", MODEL_CONFIG,
     *scheduler_args, *storage_args], cwd=REPO)
status = json.loads((OUTPUT / "run_status.json").read_text(encoding="utf-8"))
for model, result in status["models"].items():
    print(model, result["status"], "已处理:", result.get("completed_requests", 0),
          "原因:", result.get("reason", ""))
print("采集结束；下一格在 CPU 上评价，不重新推理。")


In [ ]:
# 3. 评价：CPU 读取预测，校验后生成分数、区间和全模型覆盖表
EVALUATION = OUTPUT / "evaluation"
run([PY, "-u", EXPERIMENT / "evaluate.py",
     "--results", OUTPUT, "--bank", BANK, "--output", EVALUATION,
     "--bootstrap", BOOTSTRAP_REPS, "--seed", EVALUATION_SEED], cwd=REPO)

evaluation_status = json.loads((EVALUATION / "status.json").read_text(encoding="utf-8"))
print("已评价模型:", evaluation_status["models_evaluated"],
      "未运行:", evaluation_status["models_not_run"],
      "失败或无法评价:", evaluation_status["models_unevaluable"],
      "资源/后端/授权受限:", evaluation_status["models_blocked"])
from IPython.display import display, Markdown
display(Markdown((EVALUATION / "REPORT.md").read_text(encoding="utf-8")))


In [ ]:
# 4. 下载/打包：普通 Jupyter 不依赖 google.colab
import shutil
assert (OUTPUT / "evaluation/summary.json").is_file(), "请先执行评价"

(OUTPUT / "environment-pip-freeze.txt").write_text(subprocess.check_output(
    [str(PY), "-m", "pip", "freeze"], text=True), encoding="utf-8")
if (REPO / ".git").exists():
    (OUTPUT / "repository-commit.txt").write_text(subprocess.check_output(
        ["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip(), encoding="utf-8")
    (OUTPUT / "working-tree.diff").write_text(subprocess.check_output(
        ["git", "diff", "--no-ext-diff", "HEAD"], cwd=REPO, text=True), encoding="utf-8")
shutil.copytree(BANK, OUTPUT / "question_bank", dirs_exist_ok=True)
shutil.copytree(LOG_DIR, OUTPUT / "configuration-logs", dirs_exist_ok=True)
code = OUTPUT / "collection_code"
code.mkdir(exist_ok=True)
for name in ("build.py", "run.py", "adaptive.py", "models.py", "environment.py",
             "requirements-models.txt", "requirements-notebook.txt", "evaluate.py",
             "model_cohort.json", "EVALUATION.md", "COMPATIBILITY.md",
             "requirements-runtime.txt", "sources.json", "bank_receipt.json",
             "PROTOCOL.md", "README.md", "SERVER.md", "NOTICE.md"):
    shutil.copyfile(EXPERIMENT / name, code / name)
for relative in ("system1bench/decision_models.py", "system1bench/llm_adapter.py",
                 "research/model_expansion_v2/adapter.py", "research/model_expansion_v3/adapter.py",
                 "research/startlux_transfer_v1/adapter.py"):
    dest = OUTPUT / "adapter_sources" / relative
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(REPO / relative, dest)
archive = shutil.make_archive(str(WORK_ROOT / f"hypothesis3-results-{SESSION}"), "zip", OUTPUT)
print("结果 ZIP:", archive)
if IN_COLAB:
    from google.colab import files
    files.download(archive)
else:
    from IPython.display import display, FileLink
    display(FileLink(os.path.relpath(archive, Path.cwd())))
